# Create Multi Agents to Research and Write an A
* ***AI-Related Blog Article or Research Paper Using CrewAI***

# install lib

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
%pip install "crewai[tools]"
%pip install -U google-genai


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import google.genai

print("Google GenAI SDK installed successfully")

Google GenAI SDK installed successfully


# Setup

In [4]:
import os
import asyncio
from crewai import Agent, LLM, Task, Crew, Process
from dotenv import load_dotenv

load_dotenv(override=True)
print("Environment variables loaded.")

gemini_api_key = os.getenv("GEMINI_API_KEY")

llm = LLM(
    model="gemini/gemini-3.6-flash",  # Must include 'gemini/' prefix
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.0
)

Environment variables loaded.


# Creating Agent

In [5]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True,
    llm=llm,
    max_tokens=1500
)

In [6]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True,
    llm = llm,
    max_tokens=2000
)

In [7]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True,
    llm = llm,
    max_tokens=1500
)

# Creating Tasks

***Task: Plan***

In [8]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

***Task: Write***

In [9]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

***Task: Edit***

In [10]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

# Creating the Crew

In [11]:
multiAgentCrew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)


# Running the Crew

In [12]:
import asyncio

async def run_crew():
    result = await multiAgentCrew.akickoff(inputs={'topic': 'Artificial Intelligence'})
    return result

# In Jupyter, top-level await works directly
result = await run_crew()
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3f2fac04-f861-4bcb-a882-e8f3c936b219                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: 65cf83f6-3f8a-4a4b-9bd2-6682e6e80022                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Navigating the AI Revolution: Trends, Tools, and Enterprise Strategies for 2025                              │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│                                                                                                                 │
│  Over the past 36 months, artificial intelligence has undergone an unprecedented transformation. What began as  │
│  an era of novel text-generating chatbots answering basic user prompts has rapidly evolved into a landscape     │
│  dominated by complex, multimodal systems capable of multi-step reasoning and autonomous task execution.        │
│  Today, organizations are shifting their focus from simple AI experimentation toward deep operational utility,  │
│  redefining how work gets done across every major industry sector.                                              │
│                                                                                                                 │
│  According to the latest McKinsey Global Survey on AI, enterprise adoption of these technologies has doubled    │
│  over the past two years, with more than 65% of organizations regularly deploying generative tools within at    │
│  least one core business function. As we track these accelerating **Artificial Intelligence trends**, it        │
│  becomes evident that the technology is no longer merely an optional efficiency booster, but an architectural   │
│  cornerstone for future market competitiveness.                                                                 │
│                                                                                                                 │
│  The true watershed moment of this era extends beyond AI's ability to generate realistic text or imagery; it    │
│  lies in how seamlessly these models embed themselves into complex business operations. While adoption metrics  │
│  reflect widespread interest, industry analysts emphasize that organizations moving beyond basic chatbot        │
│  deployments will gain a distinct advantage over competitors slow to integrate goal-oriented AI ecosystems.     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## The Current AI Landscape: Key Players and Infrastructure                                                    │
│                                                                                                                 │
│  The market landscape is currently defined by a dynamic interplay between frontier model developers and         │
│  underlying compute providers. Industry leaders such as OpenAI, Anthropic, and Google DeepMind continue to      │
│  push the boundaries of high-reasoning, low-hallucination models like OpenAI's o-series and Gemini 1.5 Pro.     │
│  Simultaneously, open-weight pioneers like Meta and DeepSeek are proving through benchmark studies—such as      │
│  those highlighted in Stanford University's AI Index Re

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3f2fac04-f861-4bcb-a882-e8f3c936b219                                                                       │
│  Final Output: # Navigating the AI Revolution: Trends, Tools, and Enterprise Strategies for 2025                │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│                                                                                                                 │
│  Over the past 36 months, artificial intelligence has undergone an unprecedented transformation. What began as  │
│  an era of novel text-generating chatbots answering basic user prompts has rapidly evolved into a landscape     │
│  dominated by complex, multimodal systems capable of multi-step reasoning and autonomous task execution.        │
│  Today, organizations are shifting their focus from simple AI experimentation toward deep operational utility,  │
│  redefining how work gets done across every major industry sector.                                              │
│                                                                                                                 │
│  According to the latest McKinsey Global Survey on AI, enterprise adoption of these technologies has doubled    │
│  over the past two years, with more than 65% of organizations regularly deploying generative tools within at    │
│  least one core business function. As we track these accelerating **Artificial Intelligence trends**, it        │
│  becomes evident that the technology is no longer merely an optional efficiency booster, but an architectural   │
│  cornerstone for future market competitiveness.                                                                 │
│                                                                                                                 │
│  The true watershed moment of this era extends beyond AI's ability to generate realistic text or imagery; it    │
│  lies in how seamlessly these models embed themselves into complex business operations. While adoption metrics  │
│  reflect widespread interest, industry analysts emphasize that organizations moving beyond basic chatbot        │
│  deployments will gain a distinct advantage over competitors slow to integrate goal-oriented AI ecosystems.     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## The Current AI Landscape: Key Players and Infrastructure                                                    │
│                                                                                                                 │
│  The market landscape is currently defined by a dynamic interplay between frontier model developers and         │
│  underlying compute providers. Industry leaders such as OpenAI, Anthropic, and Google DeepMind continue to      │
│  push the boundaries of high-reasoning, low-hallucination models like OpenAI's o-series and Gemini 1.5 Pro.     │
│  Simultaneously, open-weight pioneers like Meta and DeepSeek are proving through benchmark studies—such as      │
│  those highlighted in Stanford University's AI Index R

# Navigating the AI Revolution: Trends, Tools, and Enterprise Strategies for 2025

## Introduction

Over the past 36 months, artificial intelligence has undergone an unprecedented transformation. What began as an era of novel text-generating chatbots answering basic user prompts has rapidly evolved into a landscape dominated by complex, multimodal systems capable of multi-step reasoning and autonomous task execution. Today, organizations are shifting their focus from simple AI experimentation toward deep operational utility, redefining how work gets done across every major industry sector.

According to the latest McKinsey Global Survey on AI, enterprise adoption of these technologies has doubled over the past two years, with more than 65% of organizations regularly deploying generative tools within at least one core business function. As we track these accelerating **Artificial Intelligence trends**, it becomes evident that the technology is no longer merely an optional efficiency boo

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [15]:
from IPython.display import Markdown
Markdown(result.raw)

# Navigating the AI Revolution: Trends, Tools, and Enterprise Strategies for 2025

## Introduction

Over the past 36 months, artificial intelligence has undergone an unprecedented transformation. What began as an era of novel text-generating chatbots answering basic user prompts has rapidly evolved into a landscape dominated by complex, multimodal systems capable of multi-step reasoning and autonomous task execution. Today, organizations are shifting their focus from simple AI experimentation toward deep operational utility, redefining how work gets done across every major industry sector.

According to the latest McKinsey Global Survey on AI, enterprise adoption of these technologies has doubled over the past two years, with more than 65% of organizations regularly deploying generative tools within at least one core business function. As we track these accelerating **Artificial Intelligence trends**, it becomes evident that the technology is no longer merely an optional efficiency booster, but an architectural cornerstone for future market competitiveness.

The true watershed moment of this era extends beyond AI's ability to generate realistic text or imagery; it lies in how seamlessly these models embed themselves into complex business operations. While adoption metrics reflect widespread interest, industry analysts emphasize that organizations moving beyond basic chatbot deployments will gain a distinct advantage over competitors slow to integrate goal-oriented AI ecosystems.

---

## The Current AI Landscape: Key Players and Infrastructure

The market landscape is currently defined by a dynamic interplay between frontier model developers and underlying compute providers. Industry leaders such as OpenAI, Anthropic, and Google DeepMind continue to push the boundaries of high-reasoning, low-hallucination models like OpenAI's o-series and Gemini 1.5 Pro. Simultaneously, open-weight pioneers like Meta and DeepSeek are proving through benchmark studies—such as those highlighted in Stanford University's AI Index Report—that open-source architectures can achieve performance parity with proprietary closed ecosystems at a fraction of the operational cost.

Supporting this software evolution is an intensely competitive hardware and cloud infrastructure layer. Semiconductor giants like NVIDIA, alongside AMD and Qualcomm, provide the essential GPU acceleration and software-hardware ecosystems that dictate the speed of model training and inference. Meanwhile, cloud providers such as Microsoft Azure AI, Amazon Bedrock, and Google Cloud Platform serve as the primary enterprise distribution pipelines, giving organizations access to the **best AI tools for enterprise efficiency** through secure API integrations.

The rapid growth of efficient open-weight models represents a significant shift toward democratizing enterprise technology. While proprietary vendors continue to hold advantages in peak reasoning benchmarks, open-source ecosystems offer a viable alternative that lowers operational costs, prevents vendor lock-in, and accelerates localized AI deployments globally.

---

## Top Trends Shaping the Future of Artificial Intelligence

### Generative AI vs Agentic AI: The Rise of Autonomous Workflows

The debate surrounding **Generative AI vs Agentic AI** marks a fundamental shift in technical architecture. While traditional generative systems rely on direct human prompts to produce static text or code outputs, agentic AI frameworks—such as AutoGPT, CrewAI, and specialized enterprise agents—operate autonomously. These systems can break high-level goals into sequential tasks, call external software tools, execute code, and self-correct errors without continuous human intervention.

From an operational standpoint, agentic workflows are already executing end-to-end customer support escalations, performing market research, and conducting automated software debugging. This shift from passive text generation to proactive execution represents one of the most significant evolutions in enterprise software architecture since the mass migration to cloud computing.

### Multimodal Intelligence and Small Language Models

Multimodal AI has officially transitioned into the mainstream, enabling platforms to process text, voice, high-resolution video, and code simultaneously in real time without performance bottlenecks. Alongside this multimodal leap is the strategic emergence of Small Language Models (SLMs) such as Microsoft’s Phi-3 and Meta’s lightweight Llama variants. These compact models run efficiently on localized hardware, offering dramatic reductions in latency and energy consumption.

Exploring **Small Language Models business benefits** reveals that massive parameter counts are not always necessary for specialized tasks. When fine-tuned on clean, domain-specific data, localized SLMs routinely match or outperform generic foundation models. Objective industry data confirms that using targeted SLMs significantly reduces API compute overhead, offering a clear technical answer for finance and innovation leaders asking **how small language models lower operational costs**.

### AI Governance and Ethics in a Regulated World

As autonomous capabilities expand, **AI governance and ethics** have taken center stage for corporate compliance officers. The enforcement rollout of the European Union’s Artificial Intelligence Act has established a binding global precedent, categorizing AI applications into strict risk tiers ranging from minimal to unacceptable risk. Organizations globally are evaluating their data collection pipelines, algorithmic bias mitigations, and model transparency practices.

Business leaders are increasingly forced to address **what is the impact of the EU AI Act on businesses** operating across international borders. Beyond legal penalties, non-compliance threatens consumer trust and brand equity. Rather than viewing regulatory compliance purely as an operational hurdle, forward-thinking leaders treat these guidelines as essential frameworks that safeguard consumer trust, mitigate legal liability, and support long-term stability in an increasingly automated world.

---

## Overcoming Key Implementation Challenges

Despite high enthusiasm, organizations face significant friction when integrating AI systems into legacy architectures. Chief among these concerns are data privacy, intellectual property leakage, and model hallucinations. When employees feed sensitive corporate data into public model endpoints, companies expose themselves to severe security liabilities. Deploying private cloud instances or localized SLMs offers a secure alternative, ensuring sensitive business logic remains safely contained within corporate firewalls.

Another major challenge involves calculating real return on investment and eliminating model hallucinations. To ensure output reliability, leading enterprises utilize Retrieval-Augmented Generation (RAG) coupled with deterministic validation pipelines. Gartner forecasts reveal that by 2026, over 80% of enterprises will have deployed GenAI-enabled applications in production environments, but those that succeed will be the ones prioritizing deterministic accuracy over unverified demonstrations.

Many enterprise implementation failures stem not from underlying technical limitations, but from a strategic misalignment between technology choices and business goals. Deploying AI for generalized productivity improvements without defining baseline metrics or data validation protocols often results in misallocated capital and operational frustration.

---

## How to Integrate AI into Existing Business Workflows Safely

Executing a successful **enterprise AI implementation strategy** requires a disciplined, multi-stage roadmap. Organizations must first audit their operational bottlenecks to identify high-volume, low-variability tasks suitable for automation. Next, technical leaders must choose the right technology stack—balancing proprietary APIs, fine-tuned open-weight models, and on-device SLMs—while establishing rigorous data governance policies to clean internal knowledge bases before launching enterprise search or RAG systems.

Once infrastructure and governance are established, the focus shifts to controlled pilot deployment and workforce enablement. Organizations should run targeted pilot programs across cross-functional teams to gather real-world usage data and refine accuracy. Crucially, upskilling staff on effective prompt engineering, platform interfaces, and output verification ensures that human oversight remains an active safety layer throughout the automated workflow.

Successfully learning **how to integrate AI into existing business workflows safely** relies on positioning these tools as collaborative capabilities rather than direct human replacements. Organizations that combine staff upskilling with human-in-the-loop oversight consistently achieve more sustainable value than those attempting total automation without adequate validation.

---

## Conclusion and Next Steps

The current landscape of artificial intelligence presents both significant opportunities and operational complexities. Moving from basic generative models to autonomous agentic workflows and localized small language models allows businesses to drive efficiency while maintaining control over costs and security. Success in this evolving ecosystem requires a balanced approach—one that combines cutting-edge technical innovation with strict compliance and strategic human oversight.

Ultimately, navigating this transformation successfully is not about adopting every trendy tool that hits the market, but about selecting reliable systems that solve genuine business problems. By establishing robust data governance, selecting the right model architectures, and empowering workforce adaptation, organizations can confidently harness AI to secure a lasting competitive edge.

Ready to evaluate your organization's technical readiness and build a future-proof automation strategy? **Download our Enterprise AI Readiness Checklist** today to assess your data governance and technology infrastructure before implementation. For ongoing analysis and practical guides on emerging technology, **subscribe to our newsletter** for weekly insights delivered straight to your inbox.